## 05: Validation window


In [1]:
from forecast.data import load_calendar, load_visits
from forecast.validation import time_split

visits = load_visits()
train, valid, cutoff = time_split(visits)

print("cutoff:", cutoff.date())
print("train rows:", len(train), "| valid rows:", len(valid))
print("valid days:", valid["visit_date"].nunique(), "| stores in valid:", valid["air_store_id"].nunique())
print("stores in valid but not in train:", len(set(valid["air_store_id"]) - set(train["air_store_id"])))

cutoff: 2017-03-14
train rows: 224077 | valid rows: 28031
valid days: 39 | stores in valid: 826
stores in valid but not in train: 0


In [2]:
cal = load_calendar()
last_train_date = visits["visit_date"].max()

valid_cal = cal[(cal["calendar_date"] > cutoff) & (cal["calendar_date"] <= last_train_date)]
test_cal = cal[cal["calendar_date"] > last_train_date]

print("holiday days in the validation window:", int(valid_cal["holiday_flg"].sum()), "of", len(valid_cal))
print("holiday days in the test window:", int(test_cal["holiday_flg"].sum()), "of", len(test_cal))

holiday days in the validation window: 1 of 39
holiday days in the test window: 4 of 39


## Key findings
1. Split: the last training date is 2017-03-14, so the validation window is 2017-03-15 to 2017-04-22 (39 days). Training has 224,077 rows and validation has 28,031 rows (11.1% of all 252,108 rows), covering 826 of the 829 stores.
2. No cold-start stores: every store in the validation window also has training history (0 new stores). This matches the real test period, where all 821 stores have history too.
3. Holiday mismatch: the validation window has only 1 holiday day out of 39, while the test window has 4 (the Golden Week days). A model can look good on validation and still struggle on the test period, so extra checks around holidays are needed.